# Camada Bronze — `alunos`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Descrição da tabela:** Cadastro demográfico de alunos (1M registros)  
**Arquivo bruto:** `Atividade11 - alunos v2.txt`

In [1]:
import json
import sys
from pathlib import Path

# Garante que notebooks/ esteja no sys.path independente do diretório de execução
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.bronze_utils import (
    get_project_paths,
    ingest_raw_file,
    profile_bronze_table,
    save_bronze_outputs,
)

ROOT_DIR, RAW_DIR, BRONZE_DIR = get_project_paths()
print(f"Origem: {RAW_DIR / 'Atividade11 - alunos v2.txt'}")
print(f"Destino Bronze: {BRONZE_DIR}")

Origem: C:\Projects\trabalho-final-mineacao-de-dados\Bases TRabalho Final\Atividade11 - alunos v2.txt
Destino Bronze: C:\Projects\trabalho-final-mineacao-de-dados\data\bronze


In [2]:
df, meta = ingest_raw_file(
    dataset_name='alunos',
    filename='Atividade11 - alunos v2.txt',
    sep='|',
    mixed_sep=False,
    raw_dir=RAW_DIR,
)

profile = profile_bronze_table(df, meta)
parquet_path, profile_path = save_bronze_outputs(df, profile, 'alunos', BRONZE_DIR)

print(f"Linhas ingeridas: {meta['rows']:,} | Tamanho bruto: {meta['size_mb']} MB")
print(f"IDs únicos de aluno: {profile['unique_id_aluno']:,} | Duplicatas exatas: {profile['exact_duplicates']:,}")
print(f"Salvo em: {parquet_path.name} e {profile_path.name}")
df.head(5)

Linhas ingeridas: 1,000,000 | Tamanho bruto: 57.0718 MB
IDs únicos de aluno: 1,000,000 | Duplicatas exatas: 0
Salvo em: alunos.parquet e alunos_profile.json


,id_aluno,nome,idade,sexo,cidade,renda_estimada,data_nascimento,_bronze_source_file,_bronze_ingested_at
0,0000001,Matheus Martins Nascimento,24,F,Campo Grande,"3124,81",2002-05-14,Atividade11 - alunos v2.txt,2026-11-13T18:54:45Z
1,0000002,Isabela Campos,43,F,Cuiabá,"2088,72",1982-11-08,Atividade11 - alunos v2.txt,2026-11-13T18:54:45Z
2,0000003,Sofia Batista,23,M,Matão,"9394,39",2003-04-22,Atividade11 - alunos v2.txt,2026-11-13T18:54:45Z
3,0000004,Rafael Gomes,67,M,Bauru,"9348,62",1959-04-29,Atividade11 - alunos v2.txt,2026-11-13T18:54:45Z
4,0000005,Leticia Oliveira,73,M,Campo Grande,"5392,82",1953-09-05,Atividade11 - alunos v2.txt,2026-11-13T18:54:45Z


In [3]:
print("=== Campos vazios por coluna ===")
print(json.dumps(profile["empty_counts"], ensure_ascii=False, indent=2))
if profile["anomalies"]:
    print("\n=== Anomalias detectadas para tratamento na Silver ===")
    print(json.dumps(profile["anomalies"], ensure_ascii=False, indent=2))

=== Campos vazios por coluna ===
{
  "id_aluno": 0,
  "nome": 5995,
  "idade": 2078,
  "sexo": 2065,
  "cidade": 2404,
  "renda_estimada": 2018,
  "data_nascimento": 2481
}

=== Anomalias detectadas para tratamento na Silver ===
{
  "sexo_distribuicao": {
    "F": 494327,
    "M": 493326,
    "Masculino": 2106,
    "": 2065,
    "N/A": 2053,
    "X": 2051,
    " fem ": 2048,
    "feminino": 2024
  },
  "idade_menor_15_ou_maior_90": 5969,
  "renda_com_virgula": 989937,
  "mojibake_cidade_ou_nome": 47
}
